# Bagging, voting & stacking

[Random forests](random-forests.ipynb) are bagging applied to trees, baked into
one crate. But the *general* techniques for combining models apply to **any**
model — and you can build them by hand:

- **Bagging** — same model type, trained on different bootstrap samples, votes averaged.
- **Voting** — *different* model types, each votes once.
- **Stacking** — a meta-model learns how to best combine the base models.

We reuse the models from the [Classification](../02b-classification/knn-classification.ipynb)
chapter (KNN, Naive Bayes) and [logistic regression](../02-regression/logistic-regression.ipynb),
and ask: does combining them beat the best single one?

In [ ]:
:dep smartcore = { version = "0.3" }
use smartcore::linalg::basic::matrix::DenseMatrix;

// Noisy 2-class data (overlap + ~15% label noise) so ensembling can help.
// Labels are u32 (unsigned) — required by GaussianNB/KNN in smartcore.
let rows: Vec<Vec<f64>> = (0..200).map(|i| {
    let base = (i % 2) as f64 + 2.0;
    let jitter = (((i * 37) % 20) as f64 - 10.0) * 0.12;
    vec![base + jitter, ((i * 53 % 20) as f64 - 10.0) * 0.2]
}).collect();
let y: Vec<u32> = (0..200).map(|i| { let c = (i % 2) as u32; if (i * 41) % 100 < 15 { 1 - c } else { c } }).collect();

fn matrix(rows: &[Vec<f64>]) -> DenseMatrix<f64> {
    DenseMatrix::new(rows.len(), rows[0].len(), rows.iter().flatten().cloned().collect(), false)
}
// Simple chronological split helper (data alternates classes, so halves stay balanced).
fn split(rows: &[Vec<f64>], y: &[u32]) -> (Vec<Vec<f64>>, Vec<u32>, Vec<Vec<f64>>, Vec<u32>) {
    let cut = rows.len() * 7 / 10;
    (rows[..cut].to_vec(), y[..cut].to_vec(), rows[cut..].to_vec(), y[cut..].to_vec())
}
println!("{} samples ready", rows.len());

## 1. Bagging (many KNNs)

Train 15 KNN models, each on its own bootstrap resample of the training set, and
majority-vote their predictions — the general form of what a random forest does
for trees:

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::metrics::accuracy;
    use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};

    let (trx, trY, tex, teY) = split(&rows, &y);
    let k5 = || KNNClassifierParameters::default().with_k(5);
    let single = KNNClassifier::fit(&matrix(&trx), &trY, k5()).unwrap();
    let single_acc = accuracy(&teY, &single.predict(&matrix(&tex)).unwrap());

    let n_models = 15u64;
    let mut ones = vec![0u32; teY.len()];
    for seed in 0..n_models {
        let mut s = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let idx: Vec<usize> = (0..trx.len()).map(|_| { s = s.wrapping_mul(6364136223846793005).wrapping_add(1); (s >> 33) as usize % trx.len() }).collect();
        let br: Vec<Vec<f64>> = idx.iter().map(|&i| trx[i].clone()).collect();
        let by: Vec<u32> = idx.iter().map(|&i| trY[i]).collect();
        let m = KNNClassifier::fit(&matrix(&br), &by, k5()).unwrap();
        for (i, p) in m.predict(&matrix(&tex)).unwrap().iter().enumerate() { ones[i] += *p; }
    }
    let bagged: Vec<u32> = ones.iter().map(|&v| if (v as u64) * 2 >= n_models { 1 } else { 0 }).collect();
    println!("single KNN        accuracy = {:.3}", single_acc);
    println!("bagged (15 KNNs)  accuracy = {:.3}", accuracy(&teY, &bagged));
}

## 2. Voting across different models

Now combine *different* model families — logistic regression, KNN, Naive Bayes —
by majority vote, and compare the ensemble against each member:

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::metrics::accuracy;
    use smartcore::linear::logistic_regression::LogisticRegression;
    use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
    use smartcore::naive_bayes::gaussian::GaussianNB;

    let (trx, trY, tex, teY) = split(&rows, &y);
    let (mtr, mte) = (matrix(&trx), matrix(&tex));
    let p_lr = LogisticRegression::fit(&mtr, &trY, Default::default()).unwrap().predict(&mte).unwrap();
    let p_knn = KNNClassifier::fit(&mtr, &trY, KNNClassifierParameters::default().with_k(5)).unwrap().predict(&mte).unwrap();
    let p_nb = GaussianNB::fit(&mtr, &trY, Default::default()).unwrap().predict(&mte).unwrap();
    let vote: Vec<u32> = (0..teY.len()).map(|i| if p_lr[i] + p_knn[i] + p_nb[i] >= 2 { 1 } else { 0 }).collect();

    println!("logistic     accuracy = {:.3}", accuracy(&teY, &p_lr));
    println!("knn          accuracy = {:.3}", accuracy(&teY, &p_knn));
    println!("naive bayes  accuracy = {:.3}", accuracy(&teY, &p_nb));
    println!("VOTE         accuracy = {:.3}", accuracy(&teY, &vote));
}

## 3. Stacking (a meta-model)

Instead of a fixed voting rule, let a **meta-model** *learn* how to combine the
base models: use their predictions as features for a final logistic regression.

```{note}
This is a **simplified** stacking demo — proper stacking uses out-of-fold base
predictions to avoid leakage, and ideally base-model *probabilities* rather than
hard labels. We use hard labels here to keep it to the essentials.
```

In [ ]:
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::metrics::accuracy;
    use smartcore::linear::logistic_regression::LogisticRegression;
    use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
    use smartcore::naive_bayes::gaussian::GaussianNB;

    let (trx, trY, tex, teY) = split(&rows, &y);
    let (mtr, mte) = (matrix(&trx), matrix(&tex));
    // Base models.
    let lr = LogisticRegression::fit(&mtr, &trY, Default::default()).unwrap();
    let knn = KNNClassifier::fit(&mtr, &trY, KNNClassifierParameters::default().with_k(5)).unwrap();
    let nb = GaussianNB::fit(&mtr, &trY, Default::default()).unwrap();
    // Meta-features = base predictions, for train and test.
    let meta = |m: &DenseMatrix<f64>| {
        let (a, b, c) = (lr.predict(m).unwrap(), knn.predict(m).unwrap(), nb.predict(m).unwrap());
        let rows: Vec<Vec<f64>> = (0..a.len()).map(|i| vec![a[i] as f64, b[i] as f64, c[i] as f64]).collect();
        matrix(&rows)
    };
    let stacker = LogisticRegression::fit(&meta(&mtr), &trY, Default::default()).unwrap();
    let pred = stacker.predict(&meta(&mte)).unwrap();
    println!("STACKED (meta-model) accuracy = {:.3}", accuracy(&teY, &pred));
}

## Does combining help? (be honest)

The numbers above tell a nuanced story worth reading carefully:

- **Bagging** KNN roughly *tied* a single KNN. Bagging reduces **variance**, so it
  pays off for high-variance models like decision trees (that's exactly why
  [random forests](random-forests.ipynb) work) — but does little for an already
  stable, low-variance model like KNN.
- **Voting** actually did **worse than its best member**: two weaker models
  (logistic regression and Naive Bayes) outvoted the stronger KNN. Naive majority
  voting silently assumes the members are comparably good.
- **Stacking** recovered the best single-model accuracy, because the meta-model
  *learned* to trust the strong member and down-weight the weak ones.

The lesson isn't "ensembles always win" — it's that they help when members are
individually decent and make *different* mistakes, and that a **learned** combiner
(stacking) is more robust than a **fixed** rule (voting) when the members are unequal.

That closes the modelling arc — from a single [linear
regression](../02-regression/linear-regression.ipynb) to forests and ensembles.
Next: [Dimensionality Reduction](../05-dimensionality-reduction/pca.ipynb).